# clinical table

In [ ]:
from datasets import cohorts_map

def load_dataset(name):
    loader = cohorts_map[name]
    _, clinical, rfs = loader(None).get_features_labels("./")
    return clinical, rfs

In [ ]:
def get_label_dist(rfs):
    for T in (2, 5):
        print(f"{T} years")
        labels = []
        for _, rfs_p in rfs.items():
            if rfs_p["T"] is None:
                continue
            elif rfs_p["T"] < T*365:
                if rfs_p["delta"] == 1:
                    labels.append(1)
                else:
                    continue
            else:
                labels.append(0)

        print("total patients ", len(rfs))
        print("patients after label compute ", len(labels))
        for i in set(labels):
            print(f"{i} ({int(100*labels.count(i)/len(labels))}%)")

In [ ]:
import numpy as np

def print_numerical(key, clinical):
    data = [i[key] for i in clinical.values() if not(i[key] is None)]
    m, p10, p90 = int(np.mean(data)), int(np.percentile(data, 10)), int(np.percentile(data, 90))
    print(f"\t {m} [{p10}-{p90}]")

def print_categorical(key, clinical):
    data = [i[key] for i in clinical.values()]
    for i in set(data):
        print(f"\t {i}: {data.count(i)} ({100*(data.count(i)/len(data)):.1f}%)")

def demographics(clinical):
    print("age")
    print_numerical("age", clinical)

    print("GTV dose")
    print_numerical("dose", clinical)

    print("GTV volume (voxels^3)")
    print_numerical("volume", clinical)

    # "Female": 0, "Male": 1
    print("sex")
    print_categorical("sex", clinical)

    print("TNM stage")
    print_categorical("stage", clinical)

    # "RT alone": 0, "ChemoRT": 1
    print("treatment")
    print_categorical("treatment", clinical)

In [ ]:
from scipy import stats
from scipy.stats import fisher_exact
import pandas

def filter_nan(values):
    return list(filter(lambda i: not(i is None or pandas.isna(i)), values))

def split_variable_groups(var, clinical_a, clinical_b):
    group_a = [i[var] for i in clinical_a.values()]
    group_b = [i[var] for i in clinical_b.values()]
    group_a = filter_nan(group_a)
    group_b = filter_nan(group_b)
    return group_a, group_b

def numerical_pvalue(var, clinical_a, clinical_b):
    group_a, group_b = split_variable_groups(var, clinical_a, clinical_b)
    _, p_val = stats.ttest_ind(group_a, group_b)
    return p_val

def categorical_pvalue(var, clinical_a, clinical_b):
    group_a, group_b = split_variable_groups(var, clinical_a, clinical_b)
    df_a = pandas.DataFrame({'group': 'a', 'outcome': group_a})
    df_b = pandas.DataFrame({'group': 'b', 'outcome': group_b})
    df = pandas.concat([df_a, df_b])
    contingency_table = pandas.crosstab(df['group'], df['outcome'])
    _, p_value = fisher_exact(contingency_table.values)
    return p_value

def demographics_pvalue(clinical_a, clinical_b):
    for var in ("age", "dose", "volume"):
        print(var)
        print(numerical_pvalue(var, clinical_a, clinical_b))

    for var in ("sex", "stage", "treatment"):
        print(var)
        print(categorical_pvalue(var, clinical_a, clinical_b))

In [ ]:
print("radcure")
clinical_a, rfs = load_dataset("radcure")
get_label_dist(rfs)
demographics(clinical_a)

print("\n")

print("headneckpetct")
clinical_b, rfs = load_dataset("headneckpetct")
get_label_dist(rfs)
demographics(clinical_b)

print("p-values")
demographics_pvalue(clinical_a, clinical_b)

# tumor volume dummy model

In [ ]:
import random
import pandas


def load(name, T):
    clinical, rfs = load_dataset(name)
    data = []
    for p, rfs_p in rfs.items():
        try:
            if rfs_p["T"] is None:
                continue
            elif rfs_p["T"] < T*365:
                if rfs_p["delta"] == 1:
                    data.append((clinical[p]["volume"],1))
                else:
                    continue
            else:
                data.append((clinical[p]["volume"],0))
        except KeyError:
            continue

    data = list(filter(lambda i: not(i[0] is None or pandas.isna(i[0])), data))
    return data

radcure = load("radcure", 2)
print(len(radcure))
headneckpetct = load("headneckpetct", 2)
print(len(headneckpetct))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_curve
import tqdm

def zero_division(num, den):
    try:
        if den == 0:
            return 0
        else:
            return num / den
    except TypeError:
        return 0
    
def compute_metrics(split_data, t):
    y = [i[1] for i in split_data]
    y_pred = [int(i[0] > t) for i in split_data]

    cm = confusion_matrix(y, y_pred, labels=[0,1]).ravel()
    tn, fp, fn, tp = cm

    m = {"acc": accuracy_score(y, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y, y_pred),
         "f1_score": f1_score(y, y_pred, zero_division=0, labels=[0,1]),
         "specificity": zero_division(tn, (tn + fp)),
         "sensitivity": zero_division(tp, (tp + fn))}
    return m

metrics_ = {"radcure": [], "headneckpetct": []}
for _ in tqdm.trange(100):
    random.shuffle(radcure)
    n = int(0.6*len(radcure))
    train, test = radcure[:n], radcure[n:]

    clinical_values, endpoints = zip(*train)
    fpr, tpr, thresholds = roc_curve(list(endpoints), list(clinical_values))
    
    # # Youden's J statistic
    # j_scores = tpr - fpr
    
    specificity = 1 - fpr
    sensitivity = tpr
    ba = (specificity + sensitivity)/2
    
    best_idx = np.argmax(ba)
    t = thresholds[best_idx]

    metrics_["radcure"].append(compute_metrics(test, t))
    metrics_["headneckpetct"].append(compute_metrics(headneckpetct, t))

In [ ]:
import pandas
for d in metrics_.keys():
    print(d)
    metrics_df = pandas.DataFrame(metrics_[d])
    for c in metrics_df.columns:
        print(f"\t {c} {metrics_df[c].mean():.3f}")

# convert to excel with mean (std)

In [ ]:
import tqdm
import pandas
import pathlib
import json

data = []
for model_dir in tqdm.tqdm(pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\106").glob("*")):
    if not(model_dir.joinpath("metrics.csv").exists() and model_dir.joinpath("params.json").exists()):
        continue

    # get best validation loss step
    metrics = pandas.read_csv(model_dir.joinpath("metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
    
    # get step with highest validation balanced accuracy
    valid_metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "balanced_accuracy")]
    mean = valid_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()
    val_loss_step = mean[["value", "step"]]
    step = val_loss_step.loc[val_loss_step["value"].idxmax()]["step"]

    # load test metrics        
    # select metrics at step
    test_metrics = metrics[(metrics["split"] == "test") & (metrics["step"] == step)]

    # average over bootstraps
    grouped = test_metrics.groupby(["dataset", "split", "metric", "step"])
    test_metrics_mean = grouped["value"].mean().reset_index()
    test_metrics_std = grouped["value"].std().reset_index()
    test_metrics_mean['value'] = [f"{a:.3f} ({b:.3f})" for a, b in zip(test_metrics_mean['value'], test_metrics_std['value'])]
    test_metrics = test_metrics_mean

    # add name and classifier type
    with open(model_dir.joinpath("params.json"), "r") as f:
        p = json.load(f)
        test_metrics["name"] = model_dir.name
        test_metrics["backbone"] = p['backbone']
        test_metrics["task"] = p["task"]
        test_metrics["modality"] = p["modality"]
        test_metrics["pretraining"] = p["pretraining"]
        test_metrics["extractors"] = "+".join(p["extractors"])

    # add to df
    data.extend(test_metrics.to_dict(orient="records"))

In [ ]:
df = pandas.DataFrame(data)
df = df.pivot(index=["task", "pretraining", "modality", "backbone", "extractors", "name"], columns=["dataset", "metric"], values="value").sort_values(by="dataset", axis=1)

out_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\106_mean_std.xlsx"
with pandas.ExcelWriter(out_path) as writer:
    df.to_excel(writer)

# bar plots

In [1]:
import tqdm
import pandas
import pathlib
import json

data = []
for model_dir in tqdm.tqdm(pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\106").glob("*")):
    # get best validation loss step
    metrics = pandas.read_csv(model_dir.joinpath("metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
    
    # get step with highest validation balanced accuracy
    valid_metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "balanced_accuracy")]
    mean = valid_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()
    val_loss_step = mean[["value", "step"]]
    step = val_loss_step.loc[val_loss_step["value"].idxmax()]["step"]

    # load test metrics        
    # select metrics at step
    test_metrics = metrics[(metrics["split"] == "test") & (metrics["step"] == step)]

    # add name and classifier type
    with open(model_dir.joinpath("params.json"), "r") as f:
        p = json.load(f)
        test_metrics["name"] = model_dir.name
        test_metrics["backbone"] = p['backbone']
        test_metrics["task"] = p["task"]
        test_metrics["modality"] = p["modality"]
        test_metrics["pretraining"] = p["pretraining"] if p["pretraining"] else "none"
        test_metrics["extractors"] = "+".join(p["extractors"])
        
    test_metrics = test_metrics[test_metrics["metric"].isin(["auc", "balanced_accuracy", "f1_score"])]
    data.extend(test_metrics.to_dict(orient="records"))

0it [00:00, ?it/s]C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_25188\775723564.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_metrics["name"] = model_dir.name
C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_25188\775723564.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_metrics["backbone"] = p['backbone']
C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_25188\775723564.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

In [14]:
df["dataset"].unique()

array(['headneckpetct', 'radcure'], dtype=object)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas

df = pandas.DataFrame(data)
df = df[df["dataset"].isin(["radcure", "headneckpetct"])]

sns.set_theme(style="whitegrid")

# for task in df["task"].unique():
for task in ["rfs_2"]:
    print(task)
    for metric in ["auc", "balanced_accuracy", "f1_score"]:
        print(metric)

        dff = df[(df["metric"] == metric) & (df["task"] == task)]

        # 2. Create the Plot
        g = sns.catplot(
            data=dff, 
            kind="bar",
            x="extractors", 
            y="value", 
            hue="extractors",
            row="dataset",
            col="pretraining",
            errorbar="sd",
            legend=True,
            # palette="viridis",
            col_order=["none", "protonet", "cox+protonet"],
            row_order=["radcure", "headneckpetct"],
            sharex=True,
        )

        # # 3. Add Labels with extra padding to avoid error bar overlap
        # for ax in g.axes.flat:
        #     for container in ax.containers:
        #         # Increase padding to 12 or 15 to clear the 'sd' whiskers
        #         ax.bar_label(container, fmt='%.2f', padding=15, fontsize=8, fontweight='bold')

        for ax in g.axes.flat:
            ax.set_ylim(dff["value"].min(), dff["value"].max())

        for ax in g.axes.flat:
            ax.set_xticks([])

        # 5. Final Formatting
        g.set_axis_labels("", metric.upper())
        g.set_titles(template="{row_name} (pre-training={col_name})")
        g.tight_layout()
        
        plt.savefig(fr"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\{task}-{metric}.pdf")
        # plt.show()